In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_RISK_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/current_risk"
)

GOLD_CURRENT_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "gold/current_wildfire_risk"
)

GOLD_HISTORY_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "gold/wildfire_risk_history"
)

LOCATION_LOOKUP_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/zone_location_lookup"
)

print("Gold configuration loaded.")

In [0]:
location_lookup = (
    spark.read
    .format("delta")
    .load(LOCATION_LOOKUP_PATH)
)

print("Location lookup rows:", location_lookup.count())

In [0]:
current_risk = (
    spark.read
    .format("delta")
    .load(SILVER_RISK_PATH)
)

print("Current rows:", current_risk.count())

display(
    current_risk.select(
        "zone_id",
        "risk_score",
        "risk_category",
        "pipeline_run_timestamp"
    ).limit(5)
)
current_risk.select(
    "pipeline_run_timestamp"
).distinct().show(truncate=False)

In [0]:
LOCATION_LOOKUP_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/zone_location_lookup"
)

location_lookup = (
    spark.read
    .format("delta")
    .load(LOCATION_LOOKUP_PATH)
)

print("Location lookup rows:", location_lookup.count())

In [0]:
current_risk = (
    current_risk
    .drop("latitude", "longitude")
    .join(
        location_lookup,
        on="zone_id",
        how="left"
    )
)

In [0]:
current_risk = (
    current_risk
    .withColumn(
        "pipeline_run_timestamp",
        F.to_timestamp("pipeline_run_timestamp")
    )
    .withColumn(
        "weather_timestamp",
        F.to_timestamp("weather_timestamp")
    )
)

In [0]:
current_risk.select(
    "zone_id",
    "location_name",
    "zone_label",
    "latitude",
    "longitude",
    "google_maps_url"
).show(10, truncate=False)

In [0]:
from delta.tables import DeltaTable

history_exists = DeltaTable.isDeltaTable(
    spark,
    GOLD_HISTORY_PATH
)

if history_exists:
    history = (
        spark.read
        .format("delta")
        .load(GOLD_HISTORY_PATH)
    )

    print(
        "Existing history rows:",
        history.count()
    )

else:
    history = None

    print(
        "No previous Gold history found."
    )

In [0]:
# Identify the current pipeline run timestamp
run_timestamps = (
    current_risk
    .select("pipeline_run_timestamp")
    .distinct()
    .collect()
)

assert len(run_timestamps) == 1

current_run_ts = run_timestamps[0]["pipeline_run_timestamp"]

print("Current run timestamp:", current_run_ts)


if history_exists:

    history = (
        history
        .withColumn(
            "pipeline_run_timestamp",
            F.to_timestamp("pipeline_run_timestamp")
        )
    )

    # Only use snapshots OLDER than the current run
    older_history = history.filter(
        F.col("pipeline_run_timestamp") < F.lit(current_run_ts)
    )

    print("Older history rows:", older_history.count())

    window_spec = (
        Window
        .partitionBy("zone_id")
        .orderBy(
            F.col("pipeline_run_timestamp").desc()
        )
    )

    previous = (
        older_history
        .withColumn(
            "rn",
            F.row_number().over(window_spec)
        )
        .filter(F.col("rn") == 1)
        .select(
            "zone_id",
            F.col("risk_score").alias(
                "previous_risk_score"
            ),
            F.col("risk_category").alias(
                "previous_category"
            )
        )
    )

    current_gold = (
        current_risk
        .join(
            previous,
            on="zone_id",
            how="left"
        )
    )

else:

    current_gold = (
        current_risk
        .withColumn(
            "previous_risk_score",
            F.lit(None).cast("double")
        )
        .withColumn(
            "previous_category",
            F.lit(None).cast("string")
        )
    )

In [0]:
current_gold = (
    current_gold
    .withColumn(
        "risk_change",
        F.when(
            F.col(
                "previous_risk_score"
            ).isNotNull(),
            F.round(
                F.col("risk_score")
                -
                F.col(
                    "previous_risk_score"
                ),
                1
            )
        )
        .otherwise(
            F.lit(None).cast("double")
        )
    )
    .withColumn(
        "category_changed",
        F.when(
            F.col(
                "previous_category"
            ).isNull(),
            F.lit(False)
        )
        .otherwise(
            F.col("risk_category")
            !=
            F.col("previous_category")
        )
    )
)

In [0]:
current_gold = (
    current_gold
    .withColumn(
        "zone_label",
        F.col("zone_id")
    )
)

In [0]:
print(
    "Gold rows:",
    current_gold.count()
)

print(
    "Unique zones:",
    current_gold
    .select("zone_id")
    .distinct()
    .count()
)

print(
    "Missing risk scores:",
    current_gold
    .filter(
        F.col("risk_score").isNull()
    )
    .count()
)

print(
    "Invalid risk scores:",
    current_gold
    .filter(
        (F.col("risk_score") < 0)
        |
        (F.col("risk_score") > 100)
    )
    .count()
)

In [0]:
current_gold = current_gold.withColumn(
    "zone_label",
    F.concat(
        F.col("location_name"),
        F.lit(" — "),
        F.col("zone_id")
    )
)

(
    current_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_CURRENT_PATH)
)

print("Current Gold dataset published with friendly zone labels.")

In [0]:
gold_check = spark.read.format("delta").load(GOLD_CURRENT_PATH)

gold_check.select(
    "zone_id",
    "location_name",
    "zone_label"
).show(10, truncate=False)

In [0]:
history_check = (
    spark.read
    .format("delta")
    .load(GOLD_HISTORY_PATH)
)

history_check.printSchema()

In [0]:
from delta.tables import DeltaTable

if DeltaTable.isDeltaTable(spark, GOLD_HISTORY_PATH):

    history_delta = DeltaTable.forPath(
        spark,
        GOLD_HISTORY_PATH
    )

    (
        history_delta.alias("target")
        .merge(
            current_gold.alias("source"),
            """
            target.zone_id = source.zone_id
            AND
            target.pipeline_run_timestamp =
                source.pipeline_run_timestamp
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

else:

    (
        current_gold.write
        .format("delta")
        .mode("overwrite")
        .save(GOLD_HISTORY_PATH)
    )

print("Historical snapshot safely stored.")

In [0]:
history_check = (
    spark.read
    .format("delta")
    .load(GOLD_HISTORY_PATH)
)

print("History rows:", history_check.count())
print(
    "History snapshots:",
    history_check
    .select("pipeline_run_timestamp")
    .distinct()
    .count()
)

history_check.select(
    "zone_id",
    "location_name",
    "zone_label",
    "risk_score",
    "pipeline_run_timestamp"
).orderBy(
    F.col("pipeline_run_timestamp").desc()
).show(10, truncate=False)

In [0]:
gold_current = (
    spark.read
    .format("delta")
    .load(GOLD_CURRENT_PATH)
)

gold_history = (
    spark.read
    .format("delta")
    .load(GOLD_HISTORY_PATH)
)

print(
    "Current Gold rows:",
    gold_current.count()
)

print(
    "History rows:",
    gold_history.count()
)

print(
    "History snapshots:",
    gold_history
    .select(
        "pipeline_run_timestamp"
    )
    .distinct()
    .count()
)

display(
    gold_current
    .select(
        "zone_id",
        "risk_score",
        "risk_category",
        "previous_risk_score",
        "risk_change",
        "previous_category",
        "category_changed",
        "pipeline_run_timestamp"
    )
    .orderBy(
        F.col("risk_score").desc()
    )
    .limit(15)
)

In [0]:
gold_current_check = (
    spark.read
    .format("delta")
    .load(GOLD_CURRENT_PATH)
)

gold_history_check = (
    spark.read
    .format("delta")
    .load(GOLD_HISTORY_PATH)
)

gold_current_check.select(
    "pipeline_run_timestamp"
).distinct().show(truncate=False)

gold_history_check.select(
    "pipeline_run_timestamp"
).distinct().orderBy(
    "pipeline_run_timestamp"
).show(truncate=False)

print("History rows:", gold_history_check.count())

print(
    "History snapshots:",
    gold_history_check
    .select("pipeline_run_timestamp")
    .distinct()
    .count()
)

In [0]:
gold_current_check.select(
    "zone_id",
    "risk_score",
    "previous_risk_score",
    "risk_change",
    "risk_category",
    "previous_category",
    "category_changed"
).orderBy(
    F.col("risk_change").desc()
).show(20, truncate=False)

In [0]:
# 2. Write the enriched dataframe to Gold
(
    current_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_CURRENT_PATH)
)

print("Enriched Gold written.")

In [0]:
# 3. Fresh read from Gold
gold_check = (
    spark.read
    .format("delta")
    .load(GOLD_CURRENT_PATH)
)

gold_check.printSchema()

In [0]:
gold_check.select(
    "zone_id",
    "location_name",
    "zone_label",
    "risk_score",
    "risk_category",
    "risk_change",
    "google_maps_url"
).orderBy(
    F.col("risk_score").desc()
).show(15, truncate=False)